# Project 4: Music Popularity Prediction


## Problem Definition

This project will take data features collected for songs that have been on the Top 200 Weekly (Global) charts of Spotify in 2020 & 2021. The popularity of the song will be predicted using a tree-based regression model trained on these features.



The goals for the project are:

- Minimize the cross-validated ***root mean squared error ( RMSE )*** when predicting the popularity of a new song.

- Determine the importance of the features in driving the regression result.
The project will be done using tree-based regression techniques as covered in class. The hyperparameters of the trees should be carefully selected to avoid over-fitting.


There are three main challenges for this project:

1. Determining the outcome ( i.e. target ).  There is a "popularity" column.  But other columns may or may not be more appropriate indicators of popularity.

1. Choosing appropriate predictors ( i.e. features ). When building a machine learning model, we want to make sure that we consider how the model will be ultimately used. For this project, we are predicting the popularity of a new song. Therefore, we should only include the predictors we would have for a new song. It might help to imagine that the song will not be released for several weeks.

1. Data cleaning and feature engineering. Some creative cleaning and/or feature engineering may be needed to extract useful information for prediction.



Once again, be sure to go through the whole data science process and document as such in your Jupyter notebook.

The data is available AWS at https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv .



## Data Collection

### Imports

In [24]:
import duckdb
import pandas as pd
import os
from google.colab import userdata   # reads Colab Secrets
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split



### Convert CSV to Parquet

In [2]:
url = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv"
url


'https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv'

In [3]:
# File name without the extension: 'Spotify'
base_file = url.split("/")[-1].split(".")[0]
base_file

'Spotify'

In [4]:
# Read the remote CSV and write it straight to a local parquet file (lesson 102's ETL step)
# No all_varchar here: DuckDB auto-detects types, so number columns stay numbers
con = duckdb.connect()
con.execute(f"""
    COPY (
        SELECT *
        FROM read_csv('{url}')
    )
    TO '{base_file}.parquet'
    (FORMAT PARQUET);
""")

! ls -l {base_file}.parquet

-rw-r--r-- 1 root root 255181 Nov 24 12:39 Spotify.parquet


In [5]:
# First 10 rows
duckdb.query(f'select * from "{base_file}.parquet" limit 10').df()

,Index,Highest Charting Position,Number of Times Charted,Week of Highest Charting,Song Name,Streams,Artist,Artist Followers,Song ID,Genre,...,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
0,1,1,8,2021-07-23--2021-07-30,Beggin',"48,633,449",Måneskin,3377762,3Wrjm47oTz2sjIgck11l5e,"['indie rock italiano', 'italian pop']",...,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589,B
1,2,2,3,2021-07-23--2021-07-30,STAY (with Justin Bieber),"47,248,719",The Kid LAROI,2230022,5HCyWlXZPP0y6Gqq8TgA20,['australian hip hop'],...,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478,C#/Db
2,3,1,11,2021-06-25--2021-07-02,good 4 u,"40,162,559",Olivia Rodrigo,6266514,4ZtFanR9U6ndgddUvNcjcG,['pop'],...,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688,A
3,4,3,5,2021-07-02--2021-07-09,Bad Habits,"37,799,456",Ed Sheeran,83293380,6PQ88X9TkUIAUIZJHW2upE,"['pop', 'uk pop']",...,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591,B
4,5,5,1,2021-07-23--2021-07-30,INDUSTRY BABY (feat. Jack Harlow),"33,948,454",Lil Nas X,5473565,27NovPIUIRrOZoCHxABJwK,"['lgbtq+ hip hop', 'pop rap']",...,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894,D#/Eb
5,6,1,18,2021-05-07--2021-05-14,MONTERO (Call Me By Your Name),"30,071,134",Lil Nas X,5473565,67BtfxlNbhBmCDR2L2l8qd,"['lgbtq+ hip hop', 'pop rap']",...,0.61,0.508,-6.682,0.152,0.297,0.384,178.818,137876,0.758,G#/Ab
6,7,3,16,2021-05-14--2021-05-21,Kiss Me More (feat. SZA),"29,356,736",Doja Cat,8640063,748mdHapucXQri7IAO8yFK,"['dance pop', 'pop']",...,0.762,0.701,-3.541,0.0286,0.235,0.123,110.968,208867,0.742,G#/Ab
7,8,2,10,2021-06-18--2021-06-25,Todo De Ti,"26,951,613",Rauw Alejandro,6080597,4fSIb4hdOQ151TILNsSEaF,"['puerto rican pop', 'trap latino']",...,0.78,0.718,-3.605,0.0506,0.31,0.0932,127.949,199604,0.342,D#/Eb
8,9,3,8,2021-06-18--2021-06-25,Yonaguni,"25,030,128",Bad Bunny,36142273,2JPLbjOn0wPCngEot2STUS,"['latin', 'reggaeton', 'trap latino']",...,0.644,0.648,-4.601,0.118,0.276,0.135,179.951,206710,0.44,C#/Db
9,10,8,10,2021-07-02--2021-07-09,I WANNA BE YOUR SLAVE,"24,551,591",Måneskin,3377762,4pt5fDVTg5GhEvEtlz9dKk,"['indie rock italiano', 'italian pop']",...,0.75,0.608,-4.008,0.0387,0.00165,0.178,132.507,173347,0.958,C#/Db


In [16]:
# Column names and types
duckdb.query(f'describe select * from "{base_file}.parquet"').df()

,column_name,column_type,null,key,default,extra
0,Index,BIGINT,YES,None,None,None
1,Highest Charting Position,BIGINT,YES,None,None,None
2,Number of Times Charted,BIGINT,YES,None,None,None
3,Week of Highest Charting,VARCHAR,YES,None,None,None
4,Song Name,VARCHAR,YES,None,None,None
5,Streams,VARCHAR,YES,None,None,None
6,Artist,VARCHAR,YES,None,None,None
7,Artist Followers,VARCHAR,YES,None,None,None
8,Song ID,VARCHAR,YES,None,None,None
9,Genre,VARCHAR,YES,None,None,None


### Push to Hugging Face

In [6]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account

'hmsealy'

In [7]:
hf_repo = os.environ["HF_REPO"] = "Data_Science-22"
hf_repo

'Data_Science-22'

In [8]:
hf_file = os.environ["HF_FILE"] = f"{base_file}.parquet"
hf_file

'Spotify.parquet'

In [9]:
# Get the token from Colab Secrets. Only shows the first 3 and last 4 characters
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"

'hf_ ... kJTa'

In [10]:
! hf auth login --token ${HF_TOKEN}

Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [11]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"

In [12]:
print(hf_upload.stdout)

✓ Uploaded
  url: https://huggingface.co/datasets/hmsealy/Data_Science-22/commit/813520c314e9258aa44f43f25a2eb8e9eb0fb672



### Load from Hugging Face

In [13]:
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/{hf_file}"
hf_url

'https://huggingface.co/datasets/hmsealy/Data_Science-22/resolve/main/Spotify.parquet'

In [14]:
# Row count from the remote file
duckdb.sql(f'select count(1) as count from "{hf_url}"').df()

,count
0,1556


In [15]:
# Load into pandas for the rest of Pass 7
spotify_df = duckdb.sql(f'select * from "{hf_url}"').df()
spotify_df.head()

,Index,Highest Charting Position,Number of Times Charted,Week of Highest Charting,Song Name,Streams,Artist,Artist Followers,Song ID,Genre,...,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
0,1,1,8,2021-07-23--2021-07-30,Beggin',"48,633,449",Måneskin,3377762,3Wrjm47oTz2sjIgck11l5e,"['indie rock italiano', 'italian pop']",...,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589,B
1,2,2,3,2021-07-23--2021-07-30,STAY (with Justin Bieber),"47,248,719",The Kid LAROI,2230022,5HCyWlXZPP0y6Gqq8TgA20,['australian hip hop'],...,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478,C#/Db
2,3,1,11,2021-06-25--2021-07-02,good 4 u,"40,162,559",Olivia Rodrigo,6266514,4ZtFanR9U6ndgddUvNcjcG,['pop'],...,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688,A
3,4,3,5,2021-07-02--2021-07-09,Bad Habits,"37,799,456",Ed Sheeran,83293380,6PQ88X9TkUIAUIZJHW2upE,"['pop', 'uk pop']",...,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591,B
4,5,5,1,2021-07-23--2021-07-30,INDUSTRY BABY (feat. Jack Harlow),"33,948,454",Lil Nas X,5473565,27NovPIUIRrOZoCHxABJwK,"['lgbtq+ hip hop', 'pop rap']",...,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894,D#/Eb


## Data Cleaning

### Check Column Types

In [18]:
# describe
spotify_df.describe()

,Index,Highest Charting Position,Number of Times Charted
count,1556.000000,1556.000000,1556.000000
mean,778.500000,87.744216,10.668380
std,449.322824,58.147225,16.360546
min,1.000000,1.000000,1.000000
25%,389.750000,37.000000,1.000000
50%,778.500000,80.000000,4.000000
75%,1167.250000,137.000000,12.000000
max,1556.000000,200.000000,142.000000


In [19]:
# Data type of every column (object = text)
spotify_df.dtypes

,0
Index,int64
Highest Charting Position,int64
Number of Times Charted,int64
Week of Highest Charting,object
Song Name,object
Streams,object
Artist,object
Artist Followers,object
Song ID,object
Genre,object


Known before release (OK to use)
  - Danceability, Energy, Loudness, Speechiness, Acousticness, Liveness, Tempo, Duration (ms), Valence

Only known after release (don't use)
  - Streams, Highest Charting Position, Number of Times Charted, Week of Highest Charting

### Convert to Numbers and Drop Blanks

In [20]:
# Columns for Pass 7: target + audio features (all known before release)
target_col = 'Popularity'
feature_cols = ['Danceability', 'Energy', 'Loudness', 'Speechiness',
                'Acousticness', 'Liveness', 'Tempo', 'Duration (ms)', 'Valence']

# Turn text into numbers. Anything that isn't a number (like a blank) becomes NaN (null)
for col in [target_col] + feature_cols:
    spotify_df[col] = pd.to_numeric(spotify_df[col], errors='coerce')

# How many nulls did that create?
spotify_df[[target_col] + feature_cols].isnull().sum()

,0
Popularity,11
Danceability,11
Energy,11
Loudness,11
Speechiness,11
Acousticness,11
Liveness,11
Tempo,11
Duration (ms),11
Valence,11


In [21]:
# Keep only rows where the target and every feature have a value
spotify_clean = spotify_df.dropna(subset=[target_col] + feature_cols).copy()
print(spotify_df.shape, spotify_clean.shape)   # before vs after

(1556, 23) (1545, 23)


In [22]:
# Now describe should show all 10 columns
spotify_clean[[target_col] + feature_cols].describe()

,Popularity,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence
count,1545.000000,1545.000000,1545.000000,1545.000000,1545.000000,1545.000000,1545.000000,1545.000000,1545.000000,1545.000000
mean,70.089320,0.689997,0.633495,-6.348474,0.123656,0.248695,0.181202,122.811023,197940.816828,0.514704
std,15.824034,0.142444,0.161577,2.509281,0.110383,0.250326,0.144071,29.591088,47148.930420,0.227326
min,0.000000,0.150000,0.054000,-25.166000,0.023200,0.000025,0.019700,46.718000,30133.000000,0.032000
25%,65.000000,0.599000,0.532000,-7.491000,0.045600,0.048500,0.096600,97.960000,169266.000000,0.343000
50%,73.000000,0.707000,0.642000,-5.990000,0.076500,0.161000,0.124000,122.012000,193591.000000,0.512000
75%,80.000000,0.796000,0.752000,-4.711000,0.165000,0.388000,0.217000,143.860000,218902.000000,0.691000
max,100.000000,0.980000,0.970000,1.509000,0.884000,0.994000,0.962000,205.272000,588139.000000,0.979000


In [27]:
# Popularity summary: helps you judge whether an RMSE is "good" later
y_popularity.describe()

,Popularity
count,1545.000000
mean,70.089320
std,15.824034
min,0.000000
25%,65.000000
50%,73.000000
75%,80.000000
max,100.000000


## Modeling - Decision Tree

## Exploratory Data Analysis

In [26]:
# y = what we want to predict (the answer)
y_popularity = spotify_clean[target_col]

# X = the information the model uses to make its guess
X_features = spotify_clean[feature_cols]

print(X_features.shape, y_popularity.shape)   # check: same number of rows in both

(1545, 9) (1545,)


### Choose Tree Depth

In [30]:
# Try different depths. Too shallow = too simple. Too deep = memorizes the training data (overfitting)
depths = [2, 3, 4, 5, 6, 8, 10]
depth_scores = []     # will hold the average RMSE for each depth
n_search = 50         # number of splits to try for each depth

for d in depths:
    scores = np.zeros(n_search)    # empty array: one RMSE per split
    for i in range(n_search):
        X_train, X_test, y_train, y_test = train_test_split(X_features, y_popularity, test_size=0.25, random_state=i)
        model = DecisionTreeRegressor(max_depth=d, random_state=42)
        model.fit(X_train, y_train)          # learn from the training rows
        y_pred = model.predict(X_test)       # guess the test rows
        scores[i] = np.sqrt(mean_squared_error(y_test, y_pred))   # RMSE for this split
    depth_scores.append(scores.mean())       # average RMSE for this depth

# Put the results in a table
depth_df = pd.DataFrame({'Max depth': depths, 'RMSE': depth_scores})
print(depth_df)

# Pick the depth with the lowest RMSE
best_depth = depth_df.sort_values(by='RMSE')['Max depth'].iloc[0]
best_depth

   Max depth       RMSE
0          2  15.897984
1          3  16.296049
2          4  16.604057
3          5  17.040319
4          6  17.357267
5          8  18.436332
6         10  19.794587


np.int64(2)

### Repeated Cross-Validation

In [37]:
# Model 1: Decision tree, audio features
n = 100                          # number of train/test splits (lower it if it's slow)
tree_results = np.zeros(n)       # empty array: one RMSE per split

for i in range(n):
    X_train, X_test, y_train, y_test = train_test_split(X_features, y_popularity, test_size=0.25, random_state=i)
    model_tree = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
    model_tree.fit(X_train, y_train)
    y_pred = model_tree.predict(X_test)
    tree_results[i] = np.sqrt(mean_squared_error(y_test, y_pred))   # save this split's RMSE

print(f"Decision tree (max_depth={best_depth}) - mean RMSE: {tree_results.mean():.2f}")

Decision tree (max_depth=2) - mean RMSE: 15.83


### Baseline Comparison

In [39]:
# Baseline: a "model" that always guesses the average popularity of the training songs
baseline_results = np.zeros(n)

for i in range(n):
    X_train, X_test, y_train, y_test = train_test_split(X_features, y_popularity, test_size=0.25, random_state=i)
    guess = np.full(len(y_test), y_train.mean())    # same guess for every test song
    baseline_results[i] = np.sqrt(mean_squared_error(y_test, guess))

print(f"Baseline (always guess the average) - mean RMSE: {baseline_results.mean():.2f}")
print(f"Decision tree                       - mean RMSE: {tree_results.mean():.2f}")

Baseline (always guess the average) - mean RMSE: 15.75
Decision tree                       - mean RMSE: 15.83


## Conclusion